# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library, referencing all fields and record sets by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We use the `record_sets` attribute of the dataset to inspect what record sets and fields are defined. All entity references are exclusively by `@id`.

In [ ]:
# List all record sets, their `@id`s, and their fields with `@id`s.
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets were found in the Croissant schema metadata.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"Record set name: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
        print()

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame for analysis.
We use the record set and field `@id`s as identified above.

In [ ]:
# Extract data from each record set using their `@id`
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if not records:
        print(f"  No records found in record set {rs_id}.")
    else:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} records. Columns (@id):\n  {list(df.columns)}\n")
        # Show a preview for the first loaded record set only.
        print(df.head(2))
        break  # For demonstration, preview only the first found
else:
    print("No dataframes were loaded. Double-check the schema or Croissant endpoint.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records based on a numeric field (referenced by its `@id`), normalizing numeric fields, and optionally grouping data by a categorical field (`@id`).

Below, replace the `numeric_field_id` and `group_field_id` with valid `@id` values shown above for your use case.

In [ ]:
# If a dataframe was loaded above, continue with that record set.
if dataframes:
    # Use the first dataframe loaded above for demonstration
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]
    print(f"\nPerforming EDA on record set: {example_record_set_id}")

    # Inspect available columns and attempt to pick a numeric field (column name is the field `@id`)
    print("Available fields (@id):")
    print(df.columns.tolist())

    # For demonstration: attempt to guess a numeric field by checking dtype
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use the first numeric
    else:
        print("No numeric fields found.")
        numeric_field_id = None

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        print(f"\nFiltering records where {numeric_field_id} > {threshold:.2f}")
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records: {len(filtered_df)}")
        print(filtered_df.head())

        # Normalize the selected numeric field (z-score)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to pick a group field (categorical)
        non_numeric_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field_id = None
        # Ignore those with too many unique values
        for col in non_numeric_cols:
            if df[col].nunique() > 1 and df[col].nunique() < len(df) * 0.5:
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping filtered data by categorical field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical (group) field available for grouping.")
    else:
        print("No numeric field available for EDA in this record set.")
else:
    print("No dataframes loaded; nothing to analyze.")

## 5. Visualization
Visualize the distribution of a numeric field and, if available, group-wise averages by a categorical field, using only the `@id` references for fields.

Matplotlib and seaborn are used for convenient plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if we have EDA data from above
if dataframes and 'filtered_df' in locals() and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df, ci=None)
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: insufficient numeric data or no dataframes loaded.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a FAIR² Croissant dataset using the `mlcroissant` library, referencing all data entities by their schema `@id`. We identified the available record sets and their field `@id`s, loaded the data, performed basic exploratory analysis including filtering and normalization, and visualized distributions using these identifiers for reproducibility. 

You can adapt this workflow for other Croissant datasets while following the best practice of referencing all data schema elements by `@id` in your code and documentation.